# 3. Blank and laser dose

**Why.** The chopped runs give Fo lifetimes of about 1.05 ns, long for open PSII centres. Since
the leaf was cut to the laser spot (runs from `20261103_174131_chop_phi2` on), the AMBIT Fv/Fm
fell from 0.73 to 0.38-0.41 and now matches the Felix (0.35-0.39). Two things can still make both
wrong, and this notebook measures each directly:

1. **Blank (part 1).** Light reaching a detector that is not leaf fluorescence. Felix: laser
   photons scattered through RG9 + ND, fluorescence of fibre, filter or holder. AMBIT: an offset
   in s/r. In the small-leaf runs the AMBIT Fo hardly changed (0.030 -> 0.027) while Fm fell from
   0.114 to 0.046, and Phi2 came out negative from actinic 20 up: an offset, or a reading that
   depends on the LED light, would do that.
2. **Laser dose (part 2).** Whether the laser closes PSII centres in its spot. Then the "Fo" both
   instruments read with the laser on is an F' in laser light.

**How.** The Felix runs in histogram mode: micro-times from the hardware against the laser SYNC,
no chopping, gating or micro-time reconstruction. Nothing of the `ambit_chop` analysis enters the
Felix numbers here. The AMBIT runs your leaf runs' chopped protocols (part 1) or a dark protocol
with the laser unblocked by hand (part 2).

**Order.** Part 1 needs the leaf in place once (1.1), then removed (1.2-1.5). Part 2 needs the
leaf back and dark-adapted with the laser blocked, so do part 1 first. Between steps change only
what the step says: same AMBIT position, gains and currents, same ND on the SPAD, same laser
power. Write what you did in `NOTE`; it is saved with every run.

**You need** a card or shutter to block the laser beam (blocking keeps the laser warm; switching
it off changes its power and timing for a while) and an ND for the laser path (part 2, weak).

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd()))  # run the notebook from scripts/ambit-gate
import importlib

import ambit_chop as ac
import ambit_gate as ag
import blank_dose as bd

ag, ac, bd = importlib.reload(ag), importlib.reload(ac), importlib.reload(bd)

PORT = None  # e.g. "COM23" to skip the port scan
LASER_HZ = 80e6  # laser mode selector
THRESHOLD_V = 1.2  # SYNC and SPAD are 2.5 V LVTTL
GAP_US, CHOP_PERIOD_US = 150, 1000  # as in the leaf runs (2_ambit_chop)
RUNS = Path("runs")
SESSION = RUNS / (time.strftime("%Y%m%d_%H%M%S") + "_blank_dose")  # every run of this notebook goes here
NOTE = ""  # what is set up, e.g. "laser p65, OD2 on the SPAD, leaf piece 5 mm"
IRF = bd.load_irf()  # first CSV in irf-library/, for the shape comparisons (None if missing)
print("session folder:", SESSION, "| IRF:", IRF and IRF["name"])

In [ ]:
for _name in ("ambit", "felix"):
    try:
        globals()[_name].close()  # re-running must not leave a port locked
    except (KeyError, NameError):
        pass
ambit = ag.find_ambit(PORT)
felix = ag.FelixLink().open()
felix.set_thresholds(THRESHOLD_V)
print("Felix input rates:", {k: f"{v:,.0f}/s" for k, v in felix.count_rates(0.5).items()})

## Part 1. Blank

### 1.1 Leaf in place, laser on, AMBIT idle: the reference

Exactly as in the leaf runs. 10 s of Felix histograms in 100 ms windows: the leaf as the Felix
sees it under the laser.

In [ ]:
FELIX_RECS = globals().get("FELIX_RECS", {})  # condition -> run folder, kept across re-runs


def felix_rec(condition, seconds=10.0, plot=False):
    """plot=True: also the summed histogram with its FWHM (as compute_fwhm), saved as histogram_fwhm.png"""
    d = SESSION / ("felix_" + condition.replace(", ", "_").replace(" ", "_"))
    FELIX_RECS[condition] = bd.record_felix(
        felix, d, condition, seconds=seconds, note=NOTE, plot=plot, threshold_v=THRESHOLD_V, laser_hz=LASER_HZ
    )


felix_rec("leaf", plot=True)

### 1.2 Leaf removed, laser on

Take the leaf out; leave the holder, cup and AMBIT as they are. Whatever still has a laser peak is
not leaf fluorescence.

In [ ]:
felix_rec("no leaf", plot=True)  # plot the summed histogram and its FWHM

### 1.3 Leaf removed, laser blocked

Card in the laser beam. What is left is stray light and dark counts (the 20-35 kcps floor of
every run so far).

In [ ]:
felix_rec("no leaf, laser blocked", plot=True)  # plot the summed histogram and its FWHM

### 1.4 Optional: other conditions

E.g. black paper where the leaf was (the holder's reflection without a leaf), or the holder
removed. Use a new name each time; every recording shows up in 1.6.

In [ ]:
# felix_rec("black paper")

### 1.5 AMBIT blank: the AMBIT measuring the empty holder

**What you do**
1. Leave the leaf out, as in 1.2 and 1.3.
2. **Take the card out of the laser beam**, so the laser shines as it did in your leaf runs.
3. Run the cell below. It makes two AMBIT runs of about 6 s each, the same two measurements as your
   leaf series: actinic 0 (dark, saturating pulse, dark) and actinic 100 (actinic light, saturating
   pulse, actinic light, dark). The LED flashes; there is nothing else to do.
4. Optional: put the card back in, set `BLANK_TAG = "blank, laser blocked"` in the cell and run it
   again.

**Why.** With no leaf nothing fluoresces, so whatever s/r the AMBIT reports is its own offset
(measuring light reaching its detector, electronics). In 1.7 this offset is subtracted from your
leaf runs, window by window. The optional repeat shows whether the laser itself changes the AMBIT
reading.

**What you get.** For each run, the AMBIT s, r and s/r per line, and in the windows used for Fv/Fm
and Phi2 (Fo or Fs, Fm, after). Compare them with your leaf runs, where Fo was about 0.027:
- s/r near zero everywhere: no offset; the AMBIT values of the leaf runs stand as they are.
- s/r about the same in every line: a constant offset, which 1.7 subtracts.
- s/r that changes between the dark lines, the actinic line and the pulse: the AMBIT reading
  depends on its own LED light even without a leaf. That alone can make Phi2 negative.

The plot is the SPAD rate (1 ms) of the two runs. With no leaf, any counts during the lit lines are
LED light getting through the RG9, not fluorescence.

In [ ]:
BLANK_X = [0, 100, 250]  # actinic bytes, as in the leaf runs
BLANK_TAG = "blank"  # "blank, laser blocked" for the repeat with the card in
BLANK_RUNS = globals().get("BLANK_RUNS", {})  # (tag, X) -> run folder, kept across re-runs

fig, ax = plt.subplots(figsize=(10, 3.5))
for x in BLANK_X:
    prot = ac.phi2_protocol(x, fs_s=3.0, pulse_s=0.8, after_s=1.0, post_dark_s=0.5)
    d = SESSION / ("chop_" + BLANK_TAG.replace(", ", "_").replace(" ", "_") + f"_x{x}")
    ac.run_chopped(
        ambit,
        felix,
        prot,
        d,
        gap_us=GAP_US,
        period_us=CHOP_PERIOD_US,
        ttm_mode=dict(sync=False),
        threshold_v=THRESHOLD_V,
        laser_hz=LASER_HZ,
    )
    BLANK_RUNS[(BLANK_TAG, x)] = d
    print(bd.ambit_lines_report(d))
    t, r = bd.spad_rate(d)
    ax.semilogy(t, np.maximum(r, 1), lw=0.6, label=f"{BLANK_TAG}, actinic {x}")
ax.set_xlabel("time since the first SPAD photon [s]")
ax.set_ylabel("SPAD counts/s (1 ms)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### 1.6 Felix: what is not leaf fluorescence

One row per recording. All decays are rotated on the leaf's laser peak, so a blank without a peak
of its own is measured at the same micro-time. Laser-correlated = above the flat floor, from
0.5 ns before the peak on; rates are dead-time corrected (200 ns, a few % here).
- `laser/leaf` of *no leaf*: the share of the leaf's laser-correlated signal that does not come
  from the leaf. Below ~0.01 it changes neither intensities nor lifetimes noticeably.
- Shape of *no leaf* (right, drawn only when it has a peak): IRF-like = scattered laser, it adds
  a fast component and shortens mean and amplitude-weighted lifetimes; a decay of its own =
  fluorescence of glass, fibre or holder, it can lengthen them.
- `leaf - no leaf`: the leaf with the blank's laser-correlated part removed; compare its `t_mean`
  and `tau1` with the leaf row.
- *no leaf, laser blocked*: `floor` = light and dark counts that do not depend on the laser.

In [ ]:
BF = bd.blank_felix(FELIX_RECS, leaf="leaf", blank="no leaf")
print(bd.blank_felix_report(BF))
bd.plot_blank_felix(BF, IRF, save_to=SESSION / "blank_felix.png");

### 1.7 Blank-corrected Fv/Fm and Phi2 of saved leaf runs (no measurement)

**Nothing to do on the bench.** This cell measures nothing: it combines files already saved, the
blank runs of 1.5 and leaf runs you made earlier in `2_ambit_chop`. The leaf can be in or out.

**Which leaf runs.** `LEAF_RUNS` lists one leaf run per actinic level, made with the same protocol
as the blanks (`phi2_protocol(X)`, the series cell of `2_ambit_chop`). A blank only applies to leaf
runs measured with the same setup: same base under the leaf, AMBIT position, ND and laser power.
The defaults (3 Nov, 17:42 and 17:46) were measured on the old, fluorescent base, so they do not
match blanks taken on the new one. Make fresh leaf runs in `2_ambit_chop` and put their folder
names here.

**What it prints**, per actinic level:
- AMBIT: blank s/r and its share of the leaf's, per window; `raw` and `corrected` = 1 - low/Fm
  before and after subtracting the blank (low = Fo at actinic 0, Fs above). A blank of exactly
  0 only means "zero or below": the firmware turns negative readings into 0.
- Felix: the gap intensities of the leaf run, minus the *no leaf* laser-correlated rate of 1.6.
- A blank s/r that differs between the actinic line and the pulse means the AMBIT reading depends
  on its LED light with or without a leaf: that alone can make Phi2 negative.

In [ ]:
LEAF_RUNS = {0: RUNS / "20261103_174221_chop_phi2_x0", 100: RUNS / "20261103_174628_chop_phi2_x100"}
blank_cps = BF["stats"]["no leaf"]["laser_cps"] if "no leaf" in BF["stats"] else None
for (tag, x), d in sorted(BLANK_RUNS.items()):
    if x not in LEAF_RUNS:
        continue
    c = bd.ambit_blank_correct(LEAF_RUNS[x], d, felix_blank_laser_cps=blank_cps)
    a, lo = c["ambit"], c["low"]
    print(
        f"actinic {x}, {tag}: leaf {Path(LEAF_RUNS[x]).name}"
        + ("" if c["same_protocol"] else "  WARNING: protocols differ")
    )
    print(
        f"   AMBIT {lo} {a['leaf_low']:.4f}, blank {a['blank_low']:.4f} ({a['blank_share_low']:.0%});  "
        f"Fm {a['leaf_Fm']:.4f}, blank {a['blank_Fm']:.4f} ({a['blank_share_Fm']:.0%});  "
        f"1 - {lo}/Fm raw {a['raw']:.3f} -> corrected {a['corrected']:.3f}"
    )
    f = c.get("felix")
    if f:
        print(
            f"   Felix intensity raw {f['raw']:.3f}"
            + (f" -> corrected {f['corrected']:.3f}" if "corrected" in f else "")
            + f"  (lifetimes: tau_shared {f['tau_shared']:.3f}, tau_1exp {f['tau_1exp']:.3f})"
        )

## Part 2. Laser dose

### 2.1 Leaf back, laser blocked, dark adaptation

Put the leaf piece back where it was and block the laser with the card. In the earlier runs the
laser was probably on while the leaf "dark-adapted"; here it must be blocked the whole time. Every
dose run needs its own dark adaptation: the previous run exposed the leaf to the laser and to a
saturating pulse.

In [ ]:
DARK_MIN = 20  # minutes
bd.wait_minutes(DARK_MIN)

### 2.2 One dose run

The cell starts the Felix (20 ms windows) and the AMBIT, which runs measuring pulses at 20 Hz:
- `dark_s` with the laser blocked;
- **unblock the laser when the cell says so** (countdown and beep). The Felix measures the real
  laser-on moment, so a late hand costs nothing;
- `laser_s` with the laser;
- a saturating pulse (0.8 s at 255, not chopped), then 2 s dark. The AMBIT light floods the SPAD
  in the pulse, so the Felix has no laser peak there. The report prints the SPAD rate in the pulse
  and warns above 3.5 Mcps, the SPAD's rated maximum (SPAD-050: 3-3.5 Mcps, max 50 uW incident).
  The first run (strong-50p) had 23 Mcps.

`LASER_LABEL` names the condition. Suggested order, each after its own dark adaptation (2.1):
1. `"strong"`: the laser as in all runs so far.
2. `"weak"`: an ND of about OD 1 **in the laser path**; the SPAD keeps its own ND. 10x fewer laser
   photons: the Felix "first" decay then covers ~0.5 s instead of 0.1 s.
3. `"blocked"` (optional control): card stays in, `CUE = False`. Shows what the AMBIT measuring
   light alone does in the same time.

In [ ]:
LASER_LABEL = "blocked"  # "strong : 50%", "weak: 20%", "blocked"
CUE = True  # False for the control with the laser blocked throughout
DOSE = dict(dark_s=5.0, laser_s=20.0, pulse_s=0.8, post_s=2.0, f_ambit=20, window_s=0.02)
DOSE_RUNS = globals().get("DOSE_RUNS", [])

d = bd.run_dose(
    ambit,
    felix,
    SESSION / f"dose_{LASER_LABEL}_{time.strftime('%H%M%S')}",
    label=LASER_LABEL,
    cue=CUE,
    note=NOTE,
    threshold_v=THRESHOLD_V,
    laser_hz=LASER_HZ,
    **DOSE,
)
DOSE_RUNS.append(d)
r = bd.analyse_dose(d)
print(bd.dose_report(r))
bd.plot_dose(r, IRF, save_to=d / "dose.png");

### 2.3 Reading one run

- **AMBIT s/r (top left).** Before 0: the leaf with the laser blocked, its true dark Fo; no run
  so far had it. A **gradual rise after 0** (0.1 s to 10 s) is PSII closing under the laser. A
  **step within one sample** and nothing after it is laser light reaching the AMBIT detector, an
  instrument effect.
- **Felix (top right).** Laser-correlated rate per 20 ms window (grey, raw) and in kinetics bins
  (red, dead-time corrected); mean arrival time after the peak (purple). Time 0 is the window where
  the rate reaches half its steady level, so the points just before 0 are the unblocking ramp: the
  first laser photons on the dark-adapted leaf. In the first ~2 s the rate follows the card moving
  out of the beam, not the leaf; the mean arrival does not depend on how much light gets through,
  so read that one there. Orange band: the saturating pulse, where the SPAD is flooded (box: its rate).
- **Both instruments, relative to the last 5 s (bottom left).** If the laser closes centres, the
  AMBIT yield, the Felix intensity and the Felix mean arrival rise together; the dashed line is the
  AMBIT level before laser-on.
- **Decays (bottom right).** Felix decays of the first ~5000 laser photons and of the last 5 s,
  floor-subtracted and normalised, with the IRF.
- **Raw decays and fits (second figure).** The same decays as recorded (counts per 47 ps bin), the
  floor, the 1- and 2-exponential tail fits from 0.25 ns after the peak, and their residuals in
  Poisson sigma. Residuals within about +-2 and without a trend: the model describes the decay. The
  bump ~8 ns after the peak is in the IRF too (an echo of the laser pulse), not in the leaf.
- **Report.** `step` and `rise` of the AMBIT yield; AMBIT Fv/Fm from the dark level (laser blocked)
  and Fm; Felix mean arrival with its uncertainty; the SPAD rate in the pulse.

### 2.4 Compare the dose runs

Every run in `DOSE_RUNS` (this session), or folders listed by hand after a kernel restart. A rise
that grows with the laser power (strong > weak > blocked) settles the question.

In [ ]:
# DOSE_RUNS = sorted(SESSION.glob("dose_*"))      # after a kernel restart (set SESSION to the old folder)
results = [bd.analyse_dose(d) for d in DOSE_RUNS]
for r in results:
    a = r["ambit"]
    felix_txt = (
        f" | Felix first {r['initial']['seconds']:.2f} s: t_mean {r['initial']['t_mean_ns']:.3f} "
        f"+- {r['initial']['t_mean_err_ns']:.3f} ns, last {r['steady_s']:g} s: {r['steady']['t_mean_ns']:.3f} ns"
        if r["initial"] is not None
        else ""
    )
    print(
        f"{r['label']:<9} AMBIT dark {a['F_dark']['mean']:.4f}, step {100 * a['step']:+5.1f} %, rise "
        f"{100 * a['rise']:+5.1f} %, Fv/Fm (dark) {a['fvfm_dark']:.3f}, 1 - F(laser)/Fm {a['phi2_laser']:.3f}"
        + felix_txt
    )
if results:
    bd.compare_dose(results, save_to=SESSION / "dose_compare.png")

### 2.5 What the results decide

| Observation | Meaning | Next |
|---|---|---|
| *no leaf* laser/leaf below 0.01, leaf - no leaf = leaf | the Felix sees only the leaf | its lifetimes are the leaf's |
| *no leaf* has an IRF-shaped peak of a few % | scattered laser through RG9 + ND | subtract the blank decay in the analysis, or better filtering |
| AMBIT blank above ~10 % of the leaf's Fo | offset in the AMBIT | correct AMBIT Fv/Fm and Phi2 with the blank; check gains and currents |
| AMBIT blank s/r changes with the actinic level | the LED light enters the AMBIT reading | can explain the negative Phi2; to fix before the AMBIT is the reference |
| AMBIT rises after laser-on, strong >> weak and blocked | the laser closes PSII in its spot | less laser intensity at the leaf; Fo from the weak or blocked run |
| AMBIT flat after laser-on, Felix mean arrival constant | the laser does not act on the leaf | the difference is in the analysis: the code revision is next |

Send me the printed tables (or the session folder name) and we decide the next step from them.

## Close both instruments

In [ ]:
felix.close()
ambit.close()